# VedaVision — Deep Learning Benchmark (Species-ID)

**Purpose of this notebook:** this is NOT a replacement for the handcrafted botanical
feature + RF/SVM/HGB ensemble (that remains the primary Module 3 system). This is a
**controlled ablation benchmark** — a CNN transfer-learning baseline trained and
evaluated with *the exact same discipline* as the handcrafted pipeline
(leaf-level grouped split, `test_` files frozen and touched once, honest CV),
so that any accuracy gap (or lack of one) between DL and handcrafted features is a
fair, defensible comparison for the dissertation/viva — not an artifact of leakage.

**Two-stage transfer learning (per supervisor's suggestion):**
1. Stage 0 (optional, recommended if time allows) — fine-tune an ImageNet backbone on a
   large **public leaf dataset** (domain adaptation: ImageNet → "leaf domain").
2. Stage 1 — fine-tune that leaf-adapted backbone on the small VedaVision 12-species
   compound-leaf dataset (leaf domain → VedaVision domain).

If Stage 0 is skipped due to time, going straight ImageNet → VedaVision (Stage 1 only)
is still valid transfer learning — just weaker domain adaptation. Both paths are coded
below; toggle `USE_STAGE0`.

Run this top-to-bottom in Colab/Jupyter with a GPU if possible (CPU works but is slow).


In [11]:
# ------------------------------------------------------------------
# 0. Config — EDIT THESE PATHS
# ------------------------------------------------------------------
import os, random, json
import numpy as np
import tensorflow as tf
from pathlib import Path

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# Root of your species-ID dataset, same convention as preprocessing/species_id/batch_processor.py
# Resolve it robustly whether the kernel starts in the repo root or in notebooks/.
for base_dir in [Path.cwd(), Path.cwd().parent]:
    candidate_root = base_dir / "processed" / "images"
    if candidate_root.exists():
        DATA_ROOT = candidate_root
        break
else:
    raise FileNotFoundError(
        "Could not find processed/images. Checked: "
        f"{Path.cwd() / 'processed' / 'images'} and {Path.cwd().parent / 'processed' / 'images'}"
    )

# Point this at masked_raw images if you have them (fairest comparison — same background
# removal as the handcrafted pipeline), otherwise the raw resized images.
IMG_SIZE  = (224, 224)
BATCH_SIZE = 16
N_CLASSES  = 12

# Public leaf dataset for Stage 0 domain-adaptation pretraining (optional).
# Using Kaggle "aritra100/identify-the-leaf-of-the-plant" (ImageCLEF plant-ID dataset) —
# downloaded and parsed in Section 4 below (it needs XML parsing + an organ filter, not a
# plain folder-per-class layout, so PUBLIC_DATA_ROOT is set dynamically there, not here).
USE_STAGE0 = False

OUTPUT_DIR = Path("dl_benchmark_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


## 1. Build the file index — mirrors `model_training.py` split logic

Same rules as the handcrafted pipeline (see `VedaVision_Memory.md` §3):
- filename starts with `test_` → test set, **never augmented, touched once**
- everything else → train pool
- `leaf_id` (derived from filename, same physical leaf photographed top+bottom) is the
  **group key** — a leaf's top and bottom images must never be split across train/val folds.


In [12]:
import pandas as pd

def index_dataset(root: Path):
    rows = []
    for species_dir in sorted(root.iterdir()):
        if not species_dir.is_dir():
            continue
        species = species_dir.name
        for view in ["top", "bottom"]:
            view_dir = species_dir / view
            if not view_dir.exists():
                continue
            image_dirs = [view_dir / "masked_raw", view_dir / "enhanced", view_dir]
            for image_dir in image_dirs:
                if not image_dir.exists():
                    continue
                image_paths = sorted(image_dir.rglob("*.jpg"))
                if not image_paths:
                    continue
                for img_path in image_paths:
                    fname = img_path.name
                    is_test = fname.startswith("test_")
                    # leaf_id: strip test_ prefix and extension so top/bottom of the same
                    # physical leaf share a group key, e.g. test_001.jpg -> "001"
                    leaf_id = fname.replace("test_", "").rsplit(".", 1)[0]
                    rows.append({
                        "path": str(img_path),
                        "species": species,
                        "view": view,
                        "is_test": is_test,
                        "leaf_id": f"{species}_{leaf_id}",   # unique across species
                    })
    return pd.DataFrame(rows)

df = index_dataset(DATA_ROOT)
print(df.shape)
print(df.groupby(["species", "is_test"]).size().unstack(fill_value=0))

train_pool = df[~df.is_test].reset_index(drop=True)
test_df    = df[df.is_test].reset_index(drop=True)
print(f"train pool: {len(train_pool)} images | sealed test: {len(test_df)} images")


(5564, 5)
is_test            False  True 
species                        
beli                 384     80
kalawal              384     80
kasthuri_dehi        384     80
kathurupila          384     80
kattakumanjal        384     80
maha_undupiyaliya    384     80
nil_awariya          384     80
ranawara             388     80
siymbala             384     80
thunpath_kurundu     376     80
wal_bilin            384     80
wal_kollu            384     80
train pool: 4604 images | sealed test: 960 images


## 2. Leaf-level grouped train/val split

Same principle as `StratifiedGroupKFold(image_path)` in the handcrafted pipeline — here
we do a single grouped train/val split (simpler than full k-fold for a DL notebook under
time pressure), grouped by `leaf_id` so top/bottom of one leaf never straddle the split.

If you want a CV-style estimate instead of one split, wrap this in a loop over
`StratifiedGroupKFold` exactly like `model_training.py` does — same idea, just re-run
Stage 1 fine-tuning per fold. Given the deadline, one grouped split + the frozen sealed
test is the pragmatic choice.


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

le_species = sorted(df.species.unique())
species_to_idx = {s: i for i, s in enumerate(le_species)}
train_pool["y"] = train_pool.species.map(species_to_idx)

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, val_idx = next(sgkf.split(train_pool, train_pool.y, groups=train_pool.leaf_id))
tr_df = train_pool.iloc[train_idx].reset_index(drop=True)
va_df = train_pool.iloc[val_idx].reset_index(drop=True)

# sanity check: no leaf_id leakage between train and val
assert set(tr_df.leaf_id) & set(va_df.leaf_id) == set()
print(f"train: {len(tr_df)}  val: {len(va_df)}")


## 3. tf.data pipelines

Augmentation intentionally mirrors `preprocessing/shared/augmentation.py`'s **exclusions**
(no CoarseDropout / RandomCrop-style destructive ops, no strong colour shifts) so the DL
run isn't given an unfair augmentation budget the handcrafted pipeline didn't have —
that would bias the comparison in DL's favour for the wrong reason.


In [13]:
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    return img, label

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.rot90(img, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
    img = tf.image.random_brightness(img, 0.15)
    img = tf.image.random_contrast(img, 0.85, 1.15)
    img = tf.clip_by_value(img, 0.0, 255.0)
    return img, label

def make_dataset(frame, training):
    paths = frame.path.values
    labels = frame.y.values if "y" in frame else frame.species.map(species_to_idx).values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(len(frame), seed=SEED)
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(tr_df, training=True)
val_ds   = make_dataset(va_df, training=False)

test_df["y"] = test_df.species.map(species_to_idx)
test_ds  = make_dataset(test_df, training=False)   # built now, not touched until Section 6


## 4. Stage 0 — domain-adapt ImageNet backbone on a public leaf dataset (optional)

**Dataset:** [`aritra100/identify-the-leaf-of-the-plant`](https://www.kaggle.com/datasets/aritra100/identify-the-leaf-of-the-plant)
(the ImageCLEF plant-identification dataset — Yanikoglu et al. 2014). It has **no
compound/simple split** — that's fine for Stage 0, and worth stating plainly rather than
working around it:

> Stage 0's job is only to nudge ImageNet's generic filters toward "leaf-like" low/mid-level
> features (leaf-shaped silhouettes, vein-like edges, green/brown colour statistics, leaf
> surface texture) — a coarse leaf-vs-not-leaf domain shift, not compound-leaf structure
> specifically. Simple leaves still teach the backbone useful generic leaf texture/edge
> statistics for this purpose. The compound-leaf structure (rachis, leaflet arrangement)
> is something only Stage 1 fine-tuning on your actual VedaVision data can teach the
> model — Stage 0 doesn't, and can't, substitute for that.

That distinction is actually useful for your dissertation argument: it draws a clean line
between "generic leaf domain adaptation" (what a public dataset can give a CNN) and
"compound-leaf-specific structural learning" (what your handcrafted `morphology.py`
features target directly, and what a small fine-tuning set may not teach a CNN reliably).

**Two structural facts about this dataset that change how Stage 0 is built:**
1. It ships as `*.jpg` + one paired `*.xml` per image (ImageCLEF annotation format), **not**
   a folder-per-class layout — so `image_dataset_from_directory` won't work directly. The
   cells below parse the XMLs into a manifest first.
2. It's a multi-organ dataset (leaf photographs, leaf scans, flowers, fruit, stems, etc. —
   typical of ImageCLEF plant tasks). For "leaf domain adaptation" you want to **filter to
   leaf-organ images only** — the inspection cell below prints what organ/content tags
   actually exist in this download so you can set the right filter value with certainty
   rather than guessing.

At 1.28 GB / potentially hundreds of species, also **subsample** for Stage 0 — it doesn't
need to solve fine-grained species ID, just shift the backbone into the leaf domain, so
a capped subset trains much faster with no real loss for this purpose.


In [ ]:
# ------------------------------------------------------------------
# 4a. Download the public dataset (Kaggle) and inspect its structure
# ------------------------------------------------------------------
# pip install kagglehub --break-system-packages   # if not already installed
import kagglehub

PUBLIC_DATA_ROOT = kagglehub.dataset_download("aritra100/identify-the-leaf-of-the-plant")
PUBLIC_DATA_ROOT = Path(PUBLIC_DATA_ROOT)
print("Downloaded to:", PUBLIC_DATA_ROOT)

# See what's actually inside before assuming a layout
all_files = list(PUBLIC_DATA_ROOT.rglob("*"))
jpgs = [f for f in all_files if f.suffix.lower() == ".jpg"]
xmls = [f for f in all_files if f.suffix.lower() == ".xml"]
print(f"{len(jpgs)} jpg files, {len(xmls)} xml files")
print("Sample paths:")
for f in jpgs[:5]:
    print(" ", f)


In [ ]:
# ------------------------------------------------------------------
# 4b. Inspect one XML to see the real tag names/values (ImageCLEF's exact schema
#     varies by year/release, so confirm rather than assume)
# ------------------------------------------------------------------
import xml.etree.ElementTree as ET

sample_xml = xmls[0]
tree = ET.parse(sample_xml)
root = tree.getroot()

def flatten(elem, prefix=""):
    for child in elem:
        tag = f"{prefix}{child.tag}"
        if len(child) == 0:
            print(f"{tag:35s} = {child.text}")
        else:
            flatten(child, prefix=tag + ".")

print(f"Tags found in {sample_xml.name}:\n")
flatten(root)

# Once you've seen the printed tags, set these two to match what's actually there.
# Common ImageCLEF field names to look for: Content / Type / Organ (leaf vs flower vs
# fruit vs stem vs entire), ClassId / Species / Taxon (species label).
ORGAN_TAG   = "Content"     # <-- EDIT once you see the real tag name above
SPECIES_TAG = "ClassId"     # <-- EDIT once you see the real tag name above
LEAF_ORGAN_VALUES = {"Leaf", "LeafScan", "Sheet"}   # <-- EDIT to match real values seen


In [ ]:
# ------------------------------------------------------------------
# 4c. Parse all XMLs into a manifest, filter to leaf-organ images, subsample
# ------------------------------------------------------------------
import pandas as pd
from collections import Counter

def get_tag_value(root, dotted_tag):
    node = root
    for part in dotted_tag.split("."):
        found = node.find(part)
        if found is None:
            return None
        node = found
    return node.text

records = []
for xml_path in xmls:
    try:
        r = ET.parse(xml_path).getroot()
        organ = get_tag_value(r, ORGAN_TAG)
        species = get_tag_value(r, SPECIES_TAG)
        img_path = xml_path.with_suffix(".jpg")
        if img_path.exists() and species is not None:
            records.append({"image_path": str(img_path), "organ": organ, "species": species})
    except ET.ParseError:
        continue

public_df = pd.DataFrame(records)
print("Organ value counts (confirm LEAF_ORGAN_VALUES above matches these):")
print(Counter(public_df.organ))

public_df = public_df[public_df.organ.isin(LEAF_ORGAN_VALUES)].reset_index(drop=True)
print(f"\nAfter organ filter: {len(public_df)} leaf images, {public_df.species.nunique()} species")

# Subsample for Stage 0 speed — this only needs to nudge the backbone into the leaf
# domain, not reach high accuracy on this dataset itself.
MAX_CLASSES = 40
MAX_PER_CLASS = 80

top_classes = public_df.species.value_counts().head(MAX_CLASSES).index
public_df = public_df[public_df.species.isin(top_classes)]
public_df = public_df.groupby("species", group_keys=False).apply(
    lambda g: g.sample(min(len(g), MAX_PER_CLASS), random_state=SEED)
).reset_index(drop=True)

public_species_list = sorted(public_df.species.unique())
public_species_to_idx = {s: i for i, s in enumerate(public_species_list)}
public_df["y"] = public_df.species.map(public_species_to_idx)
n_public_classes = len(public_species_list)
print(f"Stage 0 training set: {len(public_df)} images, {n_public_classes} classes")

from sklearn.model_selection import train_test_split
pub_train_df, pub_val_df = train_test_split(
    public_df, test_size=0.1, stratify=public_df.y, random_state=SEED
)


In [ ]:
# ------------------------------------------------------------------
# 4d. tf.data pipeline for the public dataset (reuses load_image/augment from Section 3)
# ------------------------------------------------------------------
def make_public_dataset(frame, training):
    paths = frame.image_path.values
    labels = frame.y.values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(len(frame), seed=SEED)
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(32).prefetch(AUTOTUNE)
    return ds

public_train_ds = make_public_dataset(pub_train_df, training=True)
public_val_ds   = make_public_dataset(pub_val_df, training=False)


In [ ]:
def build_backbone(input_shape=(224, 224, 3)):
    base = tf.keras.applications.MobileNetV2(
        input_shape=input_shape, include_top=False, weights="imagenet"
    )
    return base

stage0_weights_path = OUTPUT_DIR / "stage0_backbone.weights.h5"

if USE_STAGE0:
    base = build_backbone()
    base.trainable = False   # freeze during Stage 0 head training

    stage0_model = tf.keras.Sequential([
        tf.keras.layers.Rescaling(1./127.5, offset=-1),
        base,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(n_public_classes, activation="softmax"),
    ])
    stage0_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                          loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    stage0_model.fit(public_train_ds, validation_data=public_val_ds, epochs=8)

    # unfreeze top of backbone, fine-tune briefly at low LR
    base.trainable = True
    for layer in base.layers[:-30]:
        layer.trainable = False
    stage0_model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                          loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    stage0_model.fit(public_train_ds, validation_data=public_val_ds, epochs=4)

    base.save_weights(str(stage0_weights_path))
    print("Stage 0 backbone saved ->", stage0_weights_path)
else:
    print("Skipping Stage 0 — Stage 1 will start from plain ImageNet weights.")


## 5. Stage 1 — fine-tune on the VedaVision 12-species train set

Two-phase fine-tuning (standard transfer-learning recipe):
1. Freeze backbone, train only the new classification head (few epochs, higher LR).
2. Unfreeze the top N backbone layers, fine-tune the whole thing at a low LR with
   early stopping on the held-out **validation** fold (never the sealed test).

Class weights compensate for any species-count imbalance in the train pool.


In [14]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.arange(N_CLASSES)
weights = compute_class_weight("balanced", classes=classes, y=tr_df.y.values)
class_weight = dict(zip(classes, weights))

base = build_backbone()
if USE_STAGE0 and stage0_weights_path.exists():
    base.load_weights(str(stage0_weights_path))
    print("Loaded Stage 0 leaf-domain-adapted weights.")

base.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = tf.keras.layers.Rescaling(1./127.5, offset=-1)(inputs)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.4)(x)
outputs = tf.keras.layers.Dense(N_CLASSES, activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
]

# Phase A — head only
history_a = model.fit(train_ds, validation_data=val_ds, epochs=15,
                       class_weight=class_weight, callbacks=callbacks)

# Phase B — unfreeze top of backbone, fine-tune at low LR
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history_b = model.fit(train_ds, validation_data=val_ds, epochs=20,
                       class_weight=class_weight, callbacks=callbacks)

model.save(OUTPUT_DIR / "vedavision_dl_species_model.keras")


NameError: name 'build_backbone' is not defined

## 6. Evaluate ONCE on the sealed `test_` set

This is the step that most commonly gets fudged and inflates DL accuracy in student
projects: re-running Stage 1 after peeking at test results, or picking the checkpoint
with best test accuracy instead of best **val** accuracy. Run this cell exactly once,
after training is fully finished, using the weights already restored by `EarlyStopping`
above (best on `val_ds`, not on `test_ds`).


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

y_true = test_df.y.values
y_pred_probs = model.predict(test_ds)
y_pred = np.argmax(y_pred_probs, axis=1)

test_f1_macro = f1_score(y_true, y_pred, average="macro")
print(f"SEALED TEST f1_macro: {test_f1_macro:.4f}")
print(classification_report(y_true, y_pred, target_names=le_species))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=le_species, yticklabels=le_species, cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("DL model — sealed test confusion matrix")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "dl_test_confusion_matrix.png", dpi=150)
plt.show()


## 7. Look-alike pair check — the comparison that actually matters

Overall accuracy is not the interesting number here. What tells you whether the
handcrafted botanical features are earning their place in the dissertation is **whether
the CNN does better or worse than the handcrafted ensemble specifically on the confirmed
confusable pairs** (Beli/Wal_Kollu, Kasthuri_Dehi/Thunpath_Kurundu,
Kattakumanjal/Kalawal, Wal_Bilin/Maha_Undupiyaliya, Kathurupila/Nil_Awariya,
Ranawara/Siymbala — per `VedaVision_Memory.md`). A generic CNN backbone has no inductive
bias toward venation architecture or margin serration; it's plausible it does *worse*
specifically on these pairs even if overall accuracy looks competitive.


In [ ]:
LOOKALIKE_PAIRS = [
    ("beli", "wal_kollu"),
    ("kasthuri_dehi", "thunpath_kurundu"),
    ("kattakumanjal", "kalawal"),
    ("wal_bilin", "maha_undupiyaliya"),
    ("kathurupila", "nil_awariya"),
    ("ranawara", "siymbala"),
]

def pair_accuracy(y_true, y_pred, species_list, pair):
    a, b = pair
    ia, ib = species_list.index(a), species_list.index(b)
    mask = np.isin(y_true, [ia, ib])
    if mask.sum() == 0:
        return None
    return (y_pred[mask] == y_true[mask]).mean()

for pair in LOOKALIKE_PAIRS:
    acc = pair_accuracy(y_true, y_pred, le_species, pair)
    print(f"{pair[0]:>20} vs {pair[1]:<20}: {acc}")


## 8. Statistical comparison against the handcrafted ensemble (McNemar's test)

With only ~60 sealed-test images total, a raw accuracy difference of a couple of percent
between DL and the handcrafted ensemble is **not** on its own evidence that one approach
is better — it can easily be noise at this sample size. McNemar's test on the paired
per-image correct/incorrect calls is the defensible way to say "the difference is/isn't
significant" in the dissertation, instead of eyeballing two accuracy numbers.

Export your handcrafted ensemble's sealed-test predictions (same 60 images, same order)
to `handcrafted_test_predictions.csv` with columns `image_path,y_true,y_pred`, then run:


In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

handcrafted_preds = pd.read_csv("handcrafted_test_predictions.csv")
# align order to test_df.path
handcrafted_preds = handcrafted_preds.set_index("image_path").loc[test_df.path].reset_index()

dl_correct = (y_pred == y_true)
hc_correct = (handcrafted_preds.y_pred.values == handcrafted_preds.y_true.values)

# 2x2 contingency table: [[both correct, DL correct only], [HC correct only, both wrong]]
both_correct   = np.sum(dl_correct & hc_correct)
dl_only        = np.sum(dl_correct & ~hc_correct)
hc_only        = np.sum(~dl_correct & hc_correct)
both_wrong     = np.sum(~dl_correct & ~hc_correct)

table = [[both_correct, dl_only], [hc_only, both_wrong]]
result = mcnemar(table, exact=True)
print(table)
print(f"McNemar p-value: {result.pvalue:.4f}")
print("p < 0.05 -> the accuracy difference is statistically significant, not noise.")
print("p >= 0.05 -> the two approaches are not distinguishable on this test set;")
print("             favour whichever is more interpretable/defensible -> handcrafted.")


## 9. Grad-CAM — what is the CNN actually looking at?

This is your strongest evidence for the "handcrafted features are necessary" argument
if the numbers alone come out close. If Grad-CAM shows the network attending to leaf
edges/silhouette/colour blobs rather than fine venation or margin structure — especially
on the look-alike pairs — that's a concrete, visual, viva-ready illustration of *why* a
black-box CNN on a small dataset can't reliably learn the fine botanical traits that
distinguish look-alikes, even when its aggregate accuracy number looks fine.


In [ ]:
def make_gradcam_heatmap(img_array, model, base_model, last_conv_layer_name="Conv_1"):
    grad_model = tf.keras.models.Model(
        [model.inputs], [base_model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

# Example usage on one confusable-pair test image:
example_path = test_df[test_df.species == "kattakumanjal"].path.iloc[0]
img = tf.io.read_file(example_path)
img = tf.image.decode_jpeg(img, channels=3)
img = tf.image.resize(img, IMG_SIZE)
img_array = tf.expand_dims(img, 0)

heatmap = make_gradcam_heatmap(img_array, model, base)
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1); plt.imshow(img.numpy().astype("uint8")); plt.title("Original"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(img.numpy().astype("uint8")); plt.imshow(heatmap, cmap="jet", alpha=0.5)
plt.title("Grad-CAM"); plt.axis("off")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "gradcam_example.png", dpi=150)
plt.show()


## 10. What to put in the dissertation from this notebook

- Sealed-test `f1_macro` for the DL baseline (Section 6), reported with the same honesty
  discipline as the handcrafted result (touched once, checkpoint chosen on val not test).
- Look-alike pair accuracies (Section 7) side-by-side with the handcrafted ensemble's
  15-error breakdown already documented for the sealed test.
- McNemar's p-value (Section 8) as the statistical backing for whichever direction the
  comparison goes.
- 2–3 Grad-CAM examples (Section 9), ideally one per confusable pair, as qualitative
  evidence for *why* — this is the figure a viva panel remembers.
